# Junction and spacing evidence

`on_junction` and `junction_spanning` are computed independently and carry
the coordinate system needed to audit their interpretation.

In [ ]:
#| default_exp junctions

In [ ]:
#| export
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Iterable, Mapping

In [ ]:
#| export
@dataclass(frozen=True)
class JunctionEvidence:
    coordinate_system: str
    binder_start: int
    binder_end: int
    ligation_site: int
    annotated_positions: tuple[int, ...]
    on_junction: bool
    junction_spanning: bool
    spanned_positions: tuple[int, ...]
    nearest_junction_distance: int | None

    def as_dict(self) -> dict:
        return asdict(self)

In [ ]:
#| export
def infer_junction_evidence(
    *,
    binder_start: int,
    binder_end: int,
    ligation_site: int,
    annotated_positions: Iterable[int],
    coordinate_system: str = "transcript-spliced-0-based",
) -> JunctionEvidence:
    """Compute explicit junction facts from one coordinate system.

    The binder interval is half-open `[binder_start, binder_end)`. Therefore
    a splice coordinate at the ligation site is both on-junction and spanning;
    a splice at the final excluded endpoint is not covered.
    """
    if binder_start < 0 or binder_end <= binder_start:
        raise ValueError("binder interval must satisfy 0 <= start < end")
    if not binder_start <= ligation_site < binder_end:
        raise ValueError("ligation_site must be inside the binder interval")
    positions = tuple(sorted(set(int(position) for position in annotated_positions)))
    spanned = tuple(position for position in positions if binder_start <= position < binder_end)
    nearest = min((abs(ligation_site - position) for position in positions), default=None)
    return JunctionEvidence(
        coordinate_system=coordinate_system,
        binder_start=binder_start,
        binder_end=binder_end,
        ligation_site=ligation_site,
        annotated_positions=positions,
        on_junction=ligation_site in positions,
        junction_spanning=bool(spanned),
        spanned_positions=spanned,
        nearest_junction_distance=nearest,
    )

In [ ]:
#| export
@dataclass(frozen=True)
class SpacingEvidence:
    candidate_id: str
    comparison_scope: str
    coordinate_system: str
    threshold_nt: int
    nearest_candidate_id: str | None
    nearest_distance_nt: int | None
    violates_spacing: bool

    def __post_init__(self) -> None:
        if self.threshold_nt < 0:
            raise ValueError("threshold_nt must be non-negative")
        if self.violates_spacing and (self.nearest_candidate_id is None or self.nearest_distance_nt is None):
            raise ValueError("A spacing violation requires nearest-candidate evidence")

    def as_dict(self) -> dict:
        return asdict(self)

In [ ]:
#| export
def spacing_evidence(
    candidates: Iterable[Mapping[str, object]],
    *,
    threshold_nt: int,
    comparison_scope: str = "same-panel",
    coordinate_system: str = "transcript-spliced-0-based",
) -> dict[str, SpacingEvidence]:
    """Return nearest-neighbour spacing evidence for candidates with ligation sites.

    Input records require `candidate_id` and `ligation_site`; candidates with a
    missing ligation site receive no fabricated neighbour evidence.
    """
    rows = [(str(row["candidate_id"]), row.get("ligation_site")) for row in candidates]
    output: dict[str, SpacingEvidence] = {}
    for candidate_id, position in rows:
        if position is None:
            output[candidate_id] = SpacingEvidence(candidate_id, comparison_scope, coordinate_system, threshold_nt, None, None, False)
            continue
        neighbours = [(other_id, abs(int(position) - int(other_position))) for other_id, other_position in rows
                      if other_id != candidate_id and other_position is not None]
        nearest_id, distance = min(neighbours, key=lambda item: (item[1], item[0])) if neighbours else (None, None)
        output[candidate_id] = SpacingEvidence(
            candidate_id, comparison_scope, coordinate_system, threshold_nt,
            nearest_id, distance, distance is not None and distance < threshold_nt,
        )
    return output